In [18]:
import pandas as pd
import glob

 # Import pandas for data analysis and glob to find multiple files

files = glob.glob("data/*.csv")    # Get a list of all CSV files inside the data folder


df_list = []             

for file in files:
    df = pd.read_csv(file, encoding="latin1")
    country = file.split("\\")[-1][:2]   # Windows-safe
    df["country"] = country
    df_list.append(df)

df = pd.concat(df_list, ignore_index=True)

display(df.head())


,video_id,trending_date,title,channel_title,category_id,publish_time,tags,views,likes,dislikes,comment_count,thumbnail_link,comments_disabled,ratings_disabled,video_error_or_removed,description,country
0,n1WpP7iowLc,17.14.11,Eminem - Walk On Water (Audio) ft. BeyoncÃ©,EminemVEVO,10,2017-11-10T17:00:03.000Z,"Eminem|""Walk""|""On""|""Water""|""Aftermath/Shady/In...",17158579,787425,43420,125882,https://i.ytimg.com/vi/n1WpP7iowLc/default.jpg,False,False,False,Eminem's new track Walk on Water ft. BeyoncÃ© ...,CA
1,0dBIkQ4Mz1M,17.14.11,PLUSH - Bad Unboxing Fan Mail,iDubbbzTV,23,2017-11-13T17:00:00.000Z,"plush|""bad unboxing""|""unboxing""|""fan mail""|""id...",1014651,127794,1688,13030,https://i.ytimg.com/vi/0dBIkQ4Mz1M/default.jpg,False,False,False,STill got a lot of packages. Probably will las...,CA
2,5qpjK5DgCt4,17.14.11,"Racist Superman | Rudy Mancuso, King Bach & Le...",Rudy Mancuso,23,2017-11-12T19:05:24.000Z,"racist superman|""rudy""|""mancuso""|""king""|""bach""...",3191434,146035,5339,8181,https://i.ytimg.com/vi/5qpjK5DgCt4/default.jpg,False,False,False,WATCH MY PREVIOUS VIDEO â¶ \n\nSUBSCRIBE âº ...,CA
3,d380meD0W0M,17.14.11,I Dare You: GOING BALD!?,nigahiga,24,2017-11-12T18:01:41.000Z,"ryan|""higa""|""higatv""|""nigahiga""|""i dare you""|""...",2095828,132239,1989,17518,https://i.ytimg.com/vi/d380meD0W0M/default.jpg,False,False,False,I know it's been a while since we did this sho...,CA
4,2Vv-BfVoq4g,17.14.11,Ed Sheeran - Perfect (Official Music Video),Ed Sheeran,10,2017-11-09T11:04:14.000Z,"edsheeran|""ed sheeran""|""acoustic""|""live""|""cove...",33523622,1634130,21082,85067,https://i.ytimg.com/vi/2Vv-BfVoq4g/default.jpg,False,False,False,ð§: https://ad.gt/yt-perfect\nð°: https://...,CA


In [20]:
#2. Extract all videos that have no tag
no_tag = df[(df["tags"] == "[none]") | (df["tags"].isna())]
no_tag.head()

,video_id,trending_date,title,channel_title,category_id,publish_time,tags,views,likes,dislikes,comment_count,thumbnail_link,comments_disabled,ratings_disabled,video_error_or_removed,description,country
41,JwboxqDylgg,17.14.11,Canada Soccer's Women's National Team v USA In...,Canada Soccer,17,2017-11-13T05:53:49.000Z,[none],36311,277,28,13,https://i.ytimg.com/vi/JwboxqDylgg/default.jpg,False,False,False,Canada Soccer's Women's National Team face riv...,CA
58,9B-q8h31Bpk,17.14.11,John Oliver Tackles Louis C.K. And Donald Trum...,TV Shows,22,2017-11-13T04:49:26.000Z,[none],106029,1270,101,181,https://i.ytimg.com/vi/9B-q8h31Bpk/default.jpg,False,False,False,"John Oliver on News, Politics ...",CA
78,1UE5Dq1rvUA,17.14.11,Taylor Swift Perform Ready For It - SNL,Ken Reactz,24,2017-11-12T05:18:02.000Z,[none],320964,8069,285,717,https://i.ytimg.com/vi/1UE5Dq1rvUA/default.jpg,False,False,False,Thanks for watching please subscribe and subsc...,CA
86,pmJQ4KwliX4,17.14.11,"LATEST Q POSTS: ROTHSCHILDS, HOUSE OF SAUD, lL...",James Munder,2,2017-11-12T21:25:40.000Z,[none],116820,1503,139,1066,https://i.ytimg.com/vi/pmJQ4KwliX4/default.jpg,False,False,False,https://pastebin.ca/3930472\n\nSupport My Chan...,CA
98,lHcXhBojpeQ,17.14.11,ä¸å±TVBè¦å¸ï¼ææ£10å¹´éæ¢ ç«¹é¦¬é«®å¦...,ææç¾æç,22,2017-11-12T12:49:50.000Z,[none],88061,47,58,17,https://i.ytimg.com/vi/lHcXhBojpeQ/default.jpg,False,False,False,NaN,CA


In [21]:
# 3.For each channel, total number of views
channel_views = (
    df.groupby("channel_title")["views"]
      .sum()
      .sort_values(ascending=False)
)
channel_views.head()


channel_title
ChildishGambinoVEVO     11016766510
Marvel Entertainment    10430605449
NickyJamTV               9479859505
Ozuna                    8623329509
ibighit                  8205572221
Name: views, dtype: int64

In [30]:
# 4.Save all rows with disabled comments and disabled ratings, or that have video_error_or_removed in a new dataframe called excluded, and remove those rows from the original dataframe.
excluded = df[
    (df["comments_disabled"] == True) |
    (df["ratings_disabled"] == True) |
    (df["video_error_or_removed"] == True)
]

df = df.drop(excluded.index)


In [51]:
# 5.Add a like_ratio column storing the ratio between the number of likes and of dislikes
df["like_ratio"] = df["likes"] / df["dislikes"].replace(0, pd.NA)


In [24]:
# 6. Cluster the publish time into 10-minute intervals (e.g. from 02:20 to 02:30)

df["publish_time"] = pd.to_datetime(df["publish_time"])   # Convert publish_time to datetime format

df["publish_interval"] = df["publish_time"].dt.floor("10min")        # Cluster publish times into 10-minute intervals


In [25]:
#7. For each interval, determine the number of videos, average number of likes and of dislikes.
interval_stats = df.groupby("publish_interval").agg(
    num_videos=("video_id", "count"),
    avg_likes=("likes", "mean"),
    avg_dislikes=("dislikes", "mean")  # computed how many videos were published and their average likes and dislikes
)
interval_stats.head()


,num_videos,avg_likes,avg_dislikes
publish_interval,,,
2006-07-23 08:20:00+00:00,1,459.000000,152.0000
2007-03-05 16:20:00+00:00,9,336.666667,2.0000
2007-06-25 06:50:00+00:00,12,579.833333,11.5000
2007-12-03 20:50:00+00:00,16,187.937500,15.6875
2008-01-07 21:20:00+00:00,10,99.900000,2.0000


In [26]:
# 8. For each tag, determine the number of videos
tags_series = df["tags"].str.split("|").explode()
tag_counts = tags_series.value_counts()
tag_counts.head()

tags
[none]      35518
"funny"     14834
"comedy"    11900
"2018"      10567
"news"       5653
Name: count, dtype: int64

In [23]:
#9. Find the tags with the largest number of videos
tag_counts.head(10)


tags
[none]          35518
"funny"         14834
"comedy"        11900
"2018"          10567
"news"           5653
"music"          5544
"video"          5338
"2017"           5334
"humor"          4992
"television"     4099
Name: count, dtype: int64

In [24]:
#10. For each (tag, country) pair, compute average ratio likes/dislikes
df_tags = df.copy()
df_tags["tag"] = df_tags["tags"].str.split("|")
df_tags = df_tags.explode("tag")

tag_country_ratio = (
    df_tags.groupby(["tag", "country"])["like_ratio"]
    .mean()
)
tag_country_ratio.head()


tag  country
     CA          8.618739
     DE         24.344225
     FR         16.424636
     IN          7.652585
     JP          9.894877
Name: like_ratio, dtype: object

In [33]:
#11. For each (trending_date, country) pair, the video with the largest number of views

df["trending_date"] = pd.to_datetime(df["trending_date"], format="%y.%d.%m")  # Convert trending_date to datetime

top_trending = df.loc[
    df.groupby(["trending_date", "country"])["views"].idxmax()
]     # Select most viewed video for each date and country

top_trending[["trending_date", "country", "title", "views"]].head()


,trending_date,country,title,views
4,2017-11-14,CA,Ed Sheeran - Perfect (Official Music Video),33523622
40912,2017-11-14,DE,Ed Sheeran - Perfect (Official Music Video),33523622
81895,2017-11-14,FR,Ed Sheeran - Perfect (Official Music Video),33523622
122451,2017-11-14,GB,Ed Sheeran - Perfect (Official Music Video),33523622
161378,2017-11-14,IN,Tiger Zinda Hai | Official Trailer | Salman Kh...,35885754


In [52]:
#12. Divide trending_date into three columns: year, month, day
df["year"] = df["trending_date"].dt.year
df["month"] = df["trending_date"].dt.month
df["day"] = df["trending_date"].dt.day

In [36]:
#13. For each (month, country) pair, the video with the largest number of views
top_month_country = df.loc[
    df.groupby(["month", "country"])["views"].idxmax()
]

top_month_country[["month", "country", "title", "views"]].head()


,month,country,title,views
11214,1,CA,Bruno Mars - Finesse (Remix) [Feat. Cardi B] [...,43067983
51930,1,DE,Bruno Mars - Finesse (Remix) [Feat. Cardi B] [...,37728802
92853,1,FR,Bruno Mars - Finesse (Remix) [Feat. Cardi B] [...,37728802
135208,1,GB,Bruno Mars - Finesse (Remix) [Feat. Cardi B] [...,90598955
173405,1,IN,"Taylor Swift - End Game ft. Ed Sheeran, Future",42019590


In [49]:
import json
import glob
import pandas as pd

# Find all category JSON files
category_files = glob.glob("data/*_category_id.json")

category_rows = []

for file in category_files:
    # Extract country code from filename (e.g., US_category_id.json → US)
    country = file.split("\\")[-1][:2]

    # Open and read JSON file
    with open(file, "r", encoding="utf-8") as f:
        data = json.load(f)

    # Each JSON file contains a list of categories
    for item in data["items"]:
        category_rows.append({
            "category_id": int(item["id"]),
            "category_title": item["snippet"]["title"],
            "country": country
        })

# Create categories DataFrame
cat_df = pd.DataFrame(category_rows)

cat_df.head()

,category_id,category_title,country
0,1,Film & Animation,CA
1,2,Autos & Vehicles,CA
2,10,Music,CA
3,15,Pets & Animals,CA
4,17,Sports,CA


In [50]:
#15. For each country, determine how many videos have a category that is not assignable.
df_cat = df.merge(
    cat_df,
    on=["category_id", "country"],
    how="left"
)

# Videos with missing category_title are unassignable
unassigned = df_cat[df_cat["category_title"].isna()]

# Count unassignable videos per country
unassigned_count = unassigned.groupby("country").size()

unassigned_count

country
CA      69
DE     228
FR      85
GB      90
IN      41
JP      18
KR     280
MX     149
RU    1301
dtype: int64